# Cleaning Dataset Skema 2

Notebook ini melakukan cleaning untuk `dataset/label.xlsx` dan `dataset/images`.

Output yang dibuat:
- `dataset/labels_skema2.xlsx`
- `dataset/images_skema2`


In [5]:
# Jalankan cell ini dulu agar Excel bisa dibaca beserta warna fill cell-nya.
import subprocess
import sys

try:
    import openpyxl
    print(f'openpyxl tersedia: {openpyxl.__version__}')
except ImportError:
    print('openpyxl belum tersedia. Menginstall openpyxl...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl'])
    import openpyxl
    print(f'openpyxl terpasang: {openpyxl.__version__}')


openpyxl tersedia: 3.1.5


In [6]:
from collections import Counter
from pathlib import Path
import shutil
import string

from openpyxl import load_workbook
from openpyxl.styles.colors import COLOR_INDEX

ROOT = Path.cwd()
DATASET_DIR = ROOT / 'dataset'
INPUT_LABEL_PATH = DATASET_DIR / 'label.xlsx'
OUTPUT_LABEL_PATH = DATASET_DIR / 'labels_skema2.xlsx'
IMAGES_DIR = DATASET_DIR / 'images'
OUTPUT_IMAGES_DIR = DATASET_DIR / 'images_skema2'

ALLOWED_ANNOTATIONS = {'Amenities', 'Ancillary', 'Atraction', 'Attraction', 'Accessibility'}
RED_RGB_OVERRIDES = {'FF0000', 'C00000', 'CC0000', 'E06666', 'EA9999', 'F4CCCC', 'FFC7CE'}


def normalize_text(value):
    return '' if value is None else str(value).strip()


def short_path(path):
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def find_column(ws, header_name):
    target = normalize_text(header_name).casefold()
    for cell in ws[1]:
        if normalize_text(cell.value).casefold() == target:
            return cell.column
    headers = [normalize_text(cell.value) for cell in ws[1]]
    raise ValueError(f"Kolom '{header_name}' tidak ditemukan. Header tersedia: {headers}")


def row_has_value(ws, row_idx):
    return any(
        normalize_text(ws.cell(row=row_idx, column=col_idx).value)
        for col_idx in range(1, ws.max_column + 1)
    )


def hex_to_rgb_tuple(rgb_hex):
    if not rgb_hex:
        return None
    rgb_hex = str(rgb_hex).strip().upper()
    if len(rgb_hex) == 8:
        rgb_hex = rgb_hex[-6:]
    if len(rgb_hex) != 6:
        return None
    if any(char not in string.hexdigits for char in rgb_hex):
        return None
    return tuple(int(rgb_hex[index:index + 2], 16) for index in (0, 2, 4))


def get_fill_rgb(cell):
    fill = cell.fill
    if fill is None or fill.fill_type is None:
        return None

    color = fill.fgColor
    if color is None:
        return None

    if color.type == 'rgb' and color.rgb:
        rgb = str(color.rgb)[-6:].upper()
        return rgb if hex_to_rgb_tuple(rgb) else None

    if color.type == 'indexed' and color.indexed is not None:
        try:
            raw = COLOR_INDEX[color.indexed]
        except (IndexError, TypeError):
            return None
        rgb = str(raw)[-6:].upper()
        return rgb if hex_to_rgb_tuple(rgb) else None

    return None


def is_red_fill(cell):
    rgb = get_fill_rgb(cell)
    if rgb is None:
        return False
    if rgb in RED_RGB_OVERRIDES:
        return True

    rgb_tuple = hex_to_rgb_tuple(rgb)
    if rgb_tuple is None:
        return False

    red, green, blue = rgb_tuple
    red_family = red >= 160 and (red - green) >= 25 and (red - blue) >= 15
    orange_family = green > blue + 45 and blue < 160
    return red_family and not orange_family


def prepare_output_images_dir(path):
    dataset_root = DATASET_DIR.resolve()
    target = path.resolve()
    if dataset_root not in target.parents:
        raise ValueError(f'Folder output harus berada di dalam {dataset_root}')

    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def run_cleaning_skema2():
    if not INPUT_LABEL_PATH.exists():
        raise FileNotFoundError(f'File label tidak ditemukan: {INPUT_LABEL_PATH}')
    if not IMAGES_DIR.exists():
        raise FileNotFoundError(f'Folder images tidak ditemukan: {IMAGES_DIR}')

    wb = load_workbook(INPUT_LABEL_PATH)
    ws = wb.active
    anotasi_col = find_column(ws, 'Anotasi')
    file_col = find_column(ws, 'File')

    total_rows_awal = max(ws.max_row - 1, 0)
    rows_to_delete = []
    deleted_rows = []
    fill_counter = Counter()
    deleted_annotation_counter = Counter()
    deleted_reason_counter = Counter()
    red_allowed_count = 0

    for row_idx in range(2, ws.max_row + 1):
        anotasi_cell = ws.cell(row=row_idx, column=anotasi_col)
        file_value = normalize_text(ws.cell(row=row_idx, column=file_col).value)
        anotasi_value = normalize_text(anotasi_cell.value)
        fill_rgb = get_fill_rgb(anotasi_cell) or '(no fill)'
        red_fill = is_red_fill(anotasi_cell)

        fill_counter[(fill_rgb, 'red' if red_fill else 'not red')] += 1

        delete_reason = None
        if not anotasi_value:
            delete_reason = 'anotasi kosong'
        elif red_fill and anotasi_value not in ALLOWED_ANNOTATIONS:
            delete_reason = 'fill merah dengan anotasi di luar daftar allowed'

        if delete_reason:
            rows_to_delete.append(row_idx)
            deleted_rows.append({
                'row_awal': row_idx,
                'file': file_value,
                'anotasi': anotasi_value,
                'fill_rgb': fill_rgb,
                'reason': delete_reason,
            })
            deleted_annotation_counter[anotasi_value or '(blank)'] += 1
            deleted_reason_counter[delete_reason] += 1
        elif red_fill:
            red_allowed_count += 1

    for row_idx in reversed(rows_to_delete):
        ws.delete_rows(row_idx, 1)

    wb.save(OUTPUT_LABEL_PATH)

    wb = load_workbook(OUTPUT_LABEL_PATH)
    ws = wb.active
    file_col = find_column(ws, 'File')

    image_paths = [path for path in IMAGES_DIR.iterdir() if path.is_file()]
    image_lookup = {path.name: path for path in image_paths}
    image_lookup_lower = {}
    for path in image_paths:
        image_lookup_lower.setdefault(path.name.casefold(), path)

    prepare_output_images_dir(OUTPUT_IMAGES_DIR)

    empty_rows = []
    rows_missing = []
    missing_rows = []
    rename_rows = []
    next_number = 1

    for row_idx in range(2, ws.max_row + 1):
        if not row_has_value(ws, row_idx):
            empty_rows.append(row_idx)
            continue

        raw_file = normalize_text(ws.cell(row=row_idx, column=file_col).value)
        lookup_file = Path(raw_file).name if raw_file else ''

        if not lookup_file:
            rows_missing.append(row_idx)
            continue

        source_path = image_lookup.get(lookup_file) or image_lookup_lower.get(lookup_file.casefold())
        if source_path is None:
            rows_missing.append(row_idx)
            missing_rows.append({
                'row_setelah_filter': row_idx,
                'raw_file': raw_file,
                'lookup_file': lookup_file,
                'reason': 'file tidak ditemukan di dataset/images',
            })
            continue

        extension = Path(lookup_file).suffix or source_path.suffix or '.jpg'
        new_file = f'{next_number}{extension.lower()}'
        shutil.copy2(source_path, OUTPUT_IMAGES_DIR / new_file)
        ws.cell(row=row_idx, column=file_col).value = new_file
        rename_rows.append({
            'row_setelah_filter': row_idx,
            'old_file': raw_file,
            'source_file': source_path.name,
            'new_file': new_file,
        })
        next_number += 1

    for row_idx in sorted(rows_missing + empty_rows, reverse=True):
        ws.delete_rows(row_idx, 1)

    wb.save(OUTPUT_LABEL_PATH)

    summary = {
        'input_label': short_path(INPUT_LABEL_PATH),
        'input_images_dir': short_path(IMAGES_DIR),
        'allowed_annotations': ', '.join(sorted(ALLOWED_ANNOTATIONS)),
        'total_baris_awal': total_rows_awal,
        'baris_merah_allowed_tetap_disimpan': red_allowed_count,
        'baris_dihapus_karena_anotasi_kosong': deleted_reason_counter['anotasi kosong'],
        'baris_dihapus_karena_merah_invalid': deleted_reason_counter['fill merah dengan anotasi di luar daftar allowed'],
        'total_baris_dihapus_dari_anotasi': len(deleted_rows),
        'baris_setelah_filter_anotasi': len(rename_rows) + len(missing_rows),
        'baris_dihapus_karena_file_tidak_ada': len(missing_rows),
        'total_baris_final': len(rename_rows),
        'gambar_disalin': len(rename_rows),
        'output_label': short_path(OUTPUT_LABEL_PATH),
        'output_images_dir': short_path(OUTPUT_IMAGES_DIR),
    }

    print('Rekap cleaning skema 2')
    for key, value in summary.items():
        print(f'- {key}: {value}')

    if deleted_rows:
        print('\nContoh baris yang dihapus dari kolom Anotasi:')
        for row in deleted_rows[:10]:
            print(f"  row {row['row_awal']}: {row['file']} | {row['anotasi']} | fill {row['fill_rgb']} | {row['reason']}")

    if missing_rows:
        print('\nContoh baris yang file gambarnya tidak ditemukan:')
        for row in missing_rows[:10]:
            print(f"  row {row['row_setelah_filter']}: {row['raw_file']} - {row['reason']}")

    return summary


summary = run_cleaning_skema2()


Rekap cleaning skema 2
- input_label: dataset\label.xlsx
- input_images_dir: dataset\images
- allowed_annotations: Accessibility, Amenities, Ancillary, Atraction, Attraction
- total_baris_awal: 1304
- baris_merah_allowed_tetap_disimpan: 11
- baris_dihapus_karena_anotasi_kosong: 445
- baris_dihapus_karena_merah_invalid: 164
- total_baris_dihapus_dari_anotasi: 609
- baris_setelah_filter_anotasi: 695
- baris_dihapus_karena_file_tidak_ada: 0
- total_baris_final: 695
- gambar_disalin: 695
- output_label: dataset\labels_skema2.xlsx
- output_images_dir: dataset\images_skema2

Contoh baris yang dihapus dari kolom Anotasi:
  row 6: 5.jpg | Tidak Relevan | fill FF0000 | fill merah dengan anotasi di luar daftar allowed
  row 29: 28.jpg | Balam | fill FF0000 | fill merah dengan anotasi di luar daftar allowed
  row 32: 31.jpg | Tidak Relevan | fill FF0000 | fill merah dengan anotasi di luar daftar allowed
  row 33: 32.jpg | Tidak Relevan | fill FF0000 | fill merah dengan anotasi di luar daftar allo